# Week 2: How fast is our website?
### Descriptive statistics: mean, median, percentiles, spread

**Ticket SHOP-104** (Martin Novák, CTO):
> *Customers complain that the site is slow. Grafana shows an average response time of 160 ms, which is fine. Check it.*

**How to use this notebook:** run the cells one by one (Shift+Enter). Where you see `...`, you will fill in the code during the lecture when the lecturer asks you to (🖐).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)
DATA = "https://raw.githubusercontent.com/peternather/pas-student/main/data/shopit/"

## Part 1: A first look at the data

The access log records every HTTP request: when it arrived, who sent it, which endpoint, which server handled it and how long it took. We have a random sample of 150,000 requests over 12 weeks.

In [ ]:
requests = pd.read_csv(DATA + "requests.csv", parse_dates=["ts"])
print(requests.shape)
requests.head()

In [ ]:
# Who sends the requests and how did they end?
print(requests["client"].value_counts(), "\n")
print(requests["status"].value_counts())

In [ ]:
# Missing values in each column
requests.isna().sum()

**Before computing anything, let us decide what we are actually measuring.**

- **Bots** (`client == "bot"`) are search-engine crawlers. Customers do not care how fast Googlebot got its answer. The question is "how fast is the site *for customers*", so we drop bots.
- **Missing latency** means a timeout (status 504): the load balancer cut the request off after 10 s. It is not a "meaningless missing value" but the *worst possible experience*. We put them aside for now and come back to them in part 5.

In [ ]:
humans = requests[requests["client"] == "browser"]
latency = humans["latency_ms"].dropna()
print(f"requests from people: {len(humans)}, timeouts: {humans['latency_ms'].isna().sum()}")

## Part 2: The mean and what it hides

In [ ]:
latency.mean()

Grafana does not lie: the mean really is ~160 ms. But is it a **typical** request? Let us look at the whole distribution of values in a histogram.

In [ ]:
plt.hist(latency, bins=100, range=(0, 1000))
plt.axvline(latency.mean(), color="black", linestyle="--", label="mean")
plt.xlabel("latency [ms]")
plt.ylabel("number of requests")
plt.legend()
plt.show()

The distribution is **right-skewed**: most requests are fast, but some take very long (a long right tail). Those few slow requests pull the mean up, so the mean does not describe a "typical" request.

### 🖐 Task 1
Compute the **median** (half of the requests are faster), the **95th and 99th percentile**, and the share of requests slower than the mean.
Hint: `Series.median()`, `Series.quantile(0.95)`; the comparison `latency > latency.mean()` gives True/False, and the mean of True/False is a share.

In [ ]:
median = ...
p95 = ...
p99 = ...
share_slower_than_mean = ...
print(f"median {median:.0f} ms, p95 {p95:.0f} ms, p99 {p99:.0f} ms")
print(f"slower than the mean: {share_slower_than_mean:.0%}")

**Percentile** (quantile): p95 = 378 ms means that 95 % of requests are faster and **every 20th request is slower than 378 ms**. The median is the 50th percentile.

The **empirical cumulative distribution function** (ECDF) shows all percentiles at once: for every latency *x* it gives the share of requests faster than *x*.

In [ ]:
sorted_latency = np.sort(latency)
share = np.arange(1, len(sorted_latency) + 1) / len(sorted_latency)
plt.plot(sorted_latency, share)
for q in [0.5, 0.95, 0.99]:
    plt.axhline(q, color="grey", linewidth=0.8)
    plt.axvline(latency.quantile(q), color="grey", linewidth=0.8)
plt.xlim(0, 1000)
plt.xlabel("latency [ms]")
plt.ylabel("share of requests faster")
plt.show()

### Why care about p95 when it is "only" 5 % of requests?

One ShopIT page view triggers about **10 requests** (HTML, recommendation API, cart…). If the requests are independent, the probability that **at least one** of the 10 is slower than p95 is

$$P(\text{at least one slow}) = 1 - P(\text{no slow one}) = 1 - 0.95^{10} \approx 0.40.$$

Let us check it by simulation too: 100,000 times we randomly pick 10 requests from our data.

In [ ]:
print("formula:", 1 - 0.95 ** 10)

rng = np.random.default_rng(1)
page_views = rng.choice(latency.to_numpy(), size=(100_000, 10))
print("simulation:", (page_views.max(axis=1) > p95).mean())

**About 40 % of page views** contain at least one request slower than 378 ms. That is why companies track p95 and p99 (e.g. in an SLO: "p95 < 300 ms") rather than the mean.

## Part 3: Spread – how much values differ

- **Standard deviation** (std): the typical distance from the mean, $s = \sqrt{\frac{1}{n-1}\sum (x_i - \bar x)^2}$.
- **Interquartile range** (IQR) = p75 − p25: the width of the "middle half" of the data.

Let us compare them on a small sample of 20 requests and then add a single request that took 9.9 s.

In [ ]:
def summary(x):
    x = pd.Series(x)
    return pd.Series({"mean": x.mean(), "median": x.median(), "std": x.std(),
                      "IQR": x.quantile(0.75) - x.quantile(0.25)}).round(1)

sample = latency.sample(20, random_state=7)
with_outlier = pd.concat([sample, pd.Series([9900.0])])
pd.DataFrame({"20 requests": summary(sample), "+ one 9.9 s request": summary(with_outlier)})

A single extreme value moved the mean and the standard deviation many times over; the median and IQR barely changed.

| Robust (resistant to extremes) | Sensitive to extremes |
|---|---|
| median, percentiles, IQR | mean, standard deviation, min, max |

Sensitive does not mean bad. We need the mean e.g. to compute total load (sum = mean × count). But to describe a "typical" value of skewed data, prefer robust measures.

In [ ]:
latency.describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).round(1)

## Part 4: Comparing groups – where is the site slow?

A **box plot** summarises a distribution in five numbers: the box spans p25 to p75, the line inside is the median, the "whiskers" reach the furthest value within 1.5 × IQR of the box, and the dots beyond are outliers.

In [ ]:
order = humans.groupby("endpoint")["latency_ms"].median().sort_values().index
data = [humans.loc[humans["endpoint"] == e, "latency_ms"].dropna() for e in order]
plt.boxplot(data, tick_labels=order, showfliers=False)
plt.ylabel("latency [ms]")
plt.title("Latency by endpoint (outliers hidden)")
plt.show()

### 🖐 Task 2
Build a table: for each endpoint the number of requests and the mean, median, p95 and p99 latency, sorted by p95 descending.
Hint: `groupby("endpoint")["latency_ms"].agg(...)`; write a percentile in `agg` as `lambda s: s.quantile(0.95)`.

In [ ]:
by_endpoint = ...
by_endpoint

## Part 5: Traps – bots and timeouts

How would the numbers change if we had kept the bots, or if we counted timeouts as 10 s (a lower bound on how long the customer waited)?

In [ ]:
def latency_stats(x):
    return pd.Series({"mean": x.mean(), "median": x.median(),
                      "p95": x.quantile(0.95), "p99": x.quantile(0.99)}).round(0)

pd.DataFrame({
    "people, timeouts dropped": latency_stats(latency),
    "people + bots": latency_stats(requests["latency_ms"].dropna()),
    "people, timeout = 10 s": latency_stats(humans["latency_ms"].fillna(10_000)),
})

- **Bots** are fast (they get pages from the cache), so including them makes the site look faster.
- **Timeouts** are only 0.5 % of requests, but counting them raises the mean by ~50 ms and p99 by ~200 ms. Silently dropping them (`dropna()`) hides the worst customer experiences.

Neither version is "the right one". The right thing is to **say what you did**: "bots excluded; timeouts (0.5 %) excluded, with them p99 would be 889 ms".

### A look ahead: the logarithmic axis
On a logarithmic axis the latency distribution looks almost symmetric. We will come back to why in week 4 (the log-normal distribution).

In [ ]:
plt.hist(latency, bins=np.logspace(1, 4, 80))
plt.xscale("log")
plt.xlabel("latency [ms], log scale")
plt.ylabel("number of requests")
plt.show()

## Recipe: descriptive statistics of one numeric variable

1. **What am I measuring, and for whom?** Filter out what does not belong to the question (bots) and write it down.
2. **Missing values:** how many, and *why* are they missing? Do they mean something (a timeout)?
3. **Histogram** (with a long tail also on a log axis): is the distribution symmetric or skewed?
4. **Centre:** the median for skewed data; the mean too for symmetric data.
5. **Spread:** the IQR and percentiles (p90, p95, p99); the standard deviation only for symmetric data.
6. **Groups:** box plot and a `groupby` table.
7. **Conclusion:** numbers with units, what you excluded and why, and the answer to the ticket's question.

## Reply to ticket SHOP-104
Write a reply to Martin in 3–5 sentences: is the site slow? Which number should he track instead of the mean? Where is the problem biggest?